# Bipartite Graph Creation

#### Run Notebooks

In [1]:
%%capture
# Run data wrangling nb
%run data_grace.ipynb

#### Imports

In [2]:

# Specific to network purposes
import networkx as nx
from networkx.algorithms import bipartite
import torch

#### Data Arrangement

In [ ]:
# Create df from rsvp data but drop group_id column
rsvps_df=rsvps_df.copy()
member_event=rsvps_df.drop(columns=['group_id'])


In [ ]:
# Find all the unique event ids in the events df
event_nodes = events_df.event_id.unique()

# Create a dictionary to map event ids to integers
event_nodes_dict = {v: i for i, v in enumerate(event_nodes)}

# Apply the map to the member_event df and drop the original event id column
member_event['event_id_int'] = member_event['event_id'].map(event_nodes_dict)
member_event=member_event.drop(columns=['event_id'])

# Apply same map to the events df and set the new event id column as the index
events_df['event_id_int'] = events_df['event_id'].map(event_nodes_dict)
events_df.set_index('event_id_int', inplace=True,drop=True)

In [ ]:
# Create a dictionary of dictionaries in order to eventually assign as node attributes for members
    # Pull from one-hot encoded members df
mem_dict = mem_df_new.to_dict(orient='index')

# Create a dictionary of dictionaries in order to eventually assign as node attributes for events
event_dict= events_df.to_dict(orient='index')

#### Bipartite Graph Creation

In [ ]:
# Define empty graph
G_bipartite = nx.Graph()

# Create arrays to hold nodes
member_nodes =[]
event_nodes = []

# Add member nodes
for member_id, attrs in mem_dict.items():
    G_bipartite.add_node(member_id, type='member', **attrs)
    member_nodes.append(member_id)

# Add event nodes 
for event_id, attrs in event_dict.items():
    G_bipartite.add_node(event_id, type='event', **attrs)
    event_nodes.append(event_id)

# Add edges
for _,row in member_event.iterrows():
    G_bipartite.add_edge(row['member_id'], row['event_id_int'])

In [5]:
print(G_bipartite)


Graph with 43756 nodes and 0 edges


In [6]:
A_bipartite = bipartite.biadjacency_matrix(G_bipartite, row_order=member_nodes, column_order=event_nodes)
A_bipartite_tensor = torch.tensor(A_bipartite.toarray(), dtype=torch.float)

In [7]:
A_bipartite_tensor

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [8]:
import pandas as pd
import networkx as nx
from pathlib import Path

def serialize_bipartite_artifacts(G_bipartite, member_nodes, output_dir="data"):
    """
    Serializes the bipartite graph and member nodes.
    Ensures directory existence and enforces strict string typing for node IDs.
    """
    out_path = Path.cwd() / output_dir
    out_path.mkdir(parents=True, exist_ok=True)

    # 1. Cast node attributes
    for _, data in G_bipartite.nodes(data=True):
        for key, value in data.items():
            if isinstance(value, pd.Timestamp):
                data[key] = value.isoformat()

    # 2. Cast edge attributes
    for _, _, data in G_bipartite.edges(data=True):
        for key, value in data.items():
            if isinstance(value, pd.Timestamp):
                data[key] = value.isoformat()

    # 3. Cast graph-level attributes
    for key, value in G_bipartite.graph.items():
        if isinstance(value, pd.Timestamp):
            G_bipartite.graph[key] = value.isoformat()

    # 4. Write member nodes with explicit string casting
    with open(out_path / "member_nodes.csv", 'w') as f:
        for member in member_nodes:
            f.write(f"{str(member)}\n")
            
    # 5. Serialize Graph
    nx.write_graphml(G_bipartite, out_path / "G_bipartite.graphml")

serialize_bipartite_artifacts(G_bipartite, member_nodes)

In [9]:
def inspect_node_features(G_bipartite, expected_types=2):
    """
    Prints the attributes of one node for each distinct type present in the graph.
    
    Args:
        G_bipartite (nx.Graph): The loaded bipartite graph.
        expected_types (int): The number of distinct node types to find before 
                              terminating the search. Defaults to 2 for bipartite.
    """
    seen_types = set()
    
    for node_id, attrs in G_bipartite.nodes(data=True):
        # Extract type, falling back to 'unspecified' if missing
        node_type = attrs.get('type', 'unspecified')
        
        if node_type not in seen_types:
            seen_types.add(node_type)
            
            print(f"### Node Type: {node_type.upper()}")
            print(f"ID: {node_id}")
            print("Attributes:")
            
            if not attrs:
                print("  (No attributes found)")
            else:
                for key, value in attrs.items():
                    # Print the key, value, and the underlying data type
                    print(f"  - {key}: {value} | type: {type(value).__name__}")
            
            print("-" * 40)
            
        # Early exit optimization
        if len(seen_types) >= expected_types:
            break

# Execution execution
inspect_node_features(G_bipartite)

### Node Type: EVENT
ID: 2069
Attributes:
  - type: event | type: str
  - hometown: brentwood | type: str
  - city: brentwood | type: str
  - state: tn | type: str
  - lat: 36.0 | type: float
  - lon: -86.79 | type: float
  - event_id: bfgdflytqbdc | type: str
  - group_id: 7694232 | type: int
  - time: 2015-12-23T01:00:00 | type: str
----------------------------------------
### Node Type: MEMBER
ID: 20418
Attributes:
  - type: member | type: str
  - hometown: huntington | type: str
  - city: nashville | type: str
  - state: tn | type: str
  - lat: 36.17 | type: float
  - lon: -86.72 | type: float
----------------------------------------
